# Problem Set 3, Problem 1: Linear ODEs, Euler stability, and Runge–Kutta
ROB-GY 6013 Mathematics for Robotics, Fall 2026

**Instructions.** Write code only where a cell says `TODO`, and run all cells from top to bottom. Use only NumPy and Matplotlib.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

## The model

The joint angle obeys $\ddot q + 5\dot q + 4q = 0$ with $q(0)=1$ and $\dot q(0)=0$. With the state $x=(q,\dot q)^{\mathsf T}$ this is $\dot x = f(x) = Ax$.

`x_exact(t)` computes the exact state $x(t)$ from the eigenvectors of $A$, as in part (a). Use it to check your formula for $q(t)$.

In [3]:
A = np.array([[0.0, 1.0],
              [-4.0, -5.0]])
x0 = np.array([1.0, 0.0])

def f(x):
    """The rate of change of the state x = (q, q_dot)."""
    return A @ x

def x_exact(t):
    """Exact state at time t, by the steps of part (a)."""
    lam, V = np.linalg.eig(A)                 # eigenvalues; eigenvectors are the columns of V
    c0 = np.linalg.solve(V, x0)               # split x0 = c1 v1 + c2 v2
    return (V @ (np.exp(lam * t) * c0)).real  # c_i(t) = c_i(0) exp(lam_i t), then add back

print("exact q(2) =", x_exact(2.0)[0])

exact q(2) = 0.18033522343951608


## Part (c): Euler and RK4

Complete one step of each method, as on the M7 slides:

- Euler: $x_{k+1} = x_k + h\,f(x_k)$.
- RK4: $k_1=f(x_k)$, $k_2=f(x_k+\tfrac h2k_1)$, $k_3=f(x_k+\tfrac h2k_2)$, $k_4=f(x_k+hk_3)$, and $x_{k+1}=x_k+\tfrac h6(k_1+2k_2+2k_3+k_4)$.

The test at the end of the cell should print `[ 1.  -0.4]` and then `[ 0.98298333 -0.31258333]`.

In [ ]:
def euler_step(f, x, h):
    """One Euler step of size h from the state x."""
    # TODO (c): return the state after one Euler step.
    raise NotImplementedError("part (c): Euler")

def rk4_step(f, x, h):
    """One RK4 step of size h from the state x."""
    # TODO (c): compute k1, k2, k3, k4 and return the state after one RK4 step.
    raise NotImplementedError("part (c): RK4")

# Test: one step of size 0.1 from x0.
print(euler_step(f, x0, 0.1))
print(rk4_step(f, x0, 0.1))

### Error at $T=2$

The next cell runs both methods to $T=2$ with $h=0.1, 0.05, 0.025, 0.0125$, prints the error $|q_N-q(2)|$, and saves a log-log plot of the error against $h$ as `p1-error.png`. Nothing to change here.

In [ ]:
def simulate(step, h, T):
    """States x_0, x_1, ..., x_N from x0 with N = T / h steps of the given method."""
    xs = [x0]
    for k in range(round(T / h)):
        xs.append(step(f, xs[-1], h))
    return np.array(xs)

T = 2.0
hs = [0.1, 0.05, 0.025, 0.0125]
errors = {"Euler": [], "RK4": []}
print("     h      Euler error     RK4 error")
for h in hs:
    for name, step in [("Euler", euler_step), ("RK4", rk4_step)]:
        errors[name].append(abs(simulate(step, h, T)[-1, 0] - x_exact(T)[0]))
    print(f"{h:7.4f}   {errors['Euler'][-1]:12.3e}   {errors['RK4'][-1]:12.3e}")

plt.figure(figsize=(5, 3.5))
for name in errors:
    plt.loglog(hs, errors[name], "o-", label=name)
plt.xticks(hs, [str(h) for h in hs])
plt.minorticks_off()
plt.xlabel("step size h")
plt.ylabel("error at T = 2")
plt.legend()
plt.title("Error against step size")
plt.savefig("p1-error.png", dpi=150, bbox_inches="tight")
plt.show()

## Part (d): a large step

The next cell runs both methods with $h=0.625$ to $T=5$ and saves the plot as `p1-large-step.png`. Nothing to change here.

In [ ]:
h, T = 0.625, 5.0
t = np.arange(round(T / h) + 1) * h
t_fine = np.linspace(0, T, 300)
plt.figure(figsize=(5, 3.5))
plt.plot(t_fine, [x_exact(s)[0] for s in t_fine], "k-", label="exact")
plt.plot(t, simulate(euler_step, h, T)[:, 0], "o-", label="Euler")
plt.plot(t, simulate(rk4_step, h, T)[:, 0], "s-", label="RK4")
plt.xlabel("t")
plt.ylabel("q")
plt.legend()
plt.title("Step size h = 0.625")
plt.savefig("p1-large-step.png", dpi=150, bbox_inches="tight")
plt.show()